# ⚡ 10 · FlashAttention 与 CUDA 线程模型

> AI-Infra 面试高频：**为什么标准注意力是 O(N²) 内存？如何把它变成 IO 感知（IO-aware）算子？**

- 标准注意力中间矩阵 $S=QK^{\top}$、$P=\mathrm{softmax}(S)$ 都是 $N\times N$，写回 HBM，读写量 $O(N^2)$；
- FlashAttention 靠**分块 + 在线 softmax + 重计算**把 HBM 访问降到 $O(N^2d^2/M)$（$M$=SRAM 可容纳的元素数），并用 CUDA 线程模型榨干 SRAM。

## 📋 本节要解决的问题

| # | 问题 | 答案 |
|---|---|---|
| 1 | 注意力慢/占内存在哪？ | S、P 是 N²，HBM 读写瓶颈 |
| 2 | 为什么不存 P 也能算？ | 在线 softmax 增量更新 |
| 3 | 分块后数值还准吗？ | 在线 rescale，结果与全局等价 |
| 4 | 反向怎么省内存？ | 不存 S/P，前向重计算 |
| 5 | GPU 怎么并行？ | CUDA block/线程/共享内存 |
| 6 | 长序列怎么办？ | Flash-Decoding / GQA / FA3 |
| 7 | 加速多少？ | HBM 访问 $O(N^2)$→$O(N^2d^2/M)$ |

## 1️⃣ 标准注意力的计算与内存

$$ S = QK^{\top}/\sqrt d,\quad P = \mathrm{softmax}(S),\quad O = PV $$

- $S$、$P$ 各 $N^2$ 元素；序列 8k、head 64 → 单层单头就 $8k^2=64\text{M}$ 元素 $\approx 256$ MB（fp32）；
- HBM 读写（元素数）：$S$ 写 $N^2$ + 读 $N^2$，$P$ 写 $N^2$ + 读 $N^2$ → **$4N^2$**，再加 $Q,K,V,O$ 的 $4Nd$；
- 计算量 $O(N^2 d)$ 不变，但**内存访问次数**才是瓶颈 → 注意力是「**带宽受限**」算子。

## 2️⃣ 显存层级与 Roofline

```
HBM(几十GB, ~2TB/s)  ← 慢、大
  ↓ 按块搬
SRAM 共享内存(几十~几百KB/块, ~20TB/s)  ← 快、小
  ↓
寄存器 / ALU
```

- **Roofline**：一个 kernel 的性能上限 = 算术强度（FLOP/Byte）与硬件峰值算力/带宽的交点；
- 注意力 FLOP 不小但 N² 中间量让它在「带宽区」——**减少 HBM 往返 = 提速**；
- FlashAttention 的哲学：**能算的不存，能常驻 SRAM 的不落 HBM**。

In [1]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
import numpy as np

def hbm_access_std(N, d):
    # 标准注意力 HBM 元素访问：S写+读 + P写+读 = 4N²；QKVO 各 Nd 次读/写 ≈ 4Nd
    return 4*N**2 + 4*N*d, 4*N**2

def hbm_access_fa(N, d, M):
    # FA 口径：M = SRAM 可容纳的元素数；Q/K/V 三个 tile 同时驻留 → 3·B·d ≤ M
    # B = M/(3d) 为行列块大小；K/V 每个 Q 块被重读 N/B 次 → 2N²d/B ≈ 6N²d²/M
    B = M/(3*d)
    return 2*N*d + 2*N**2*d/B, 2*N**2*d/B   # Q 读 + O 写各 1 次；K/V 分块重读

N, d = 8192, 64
M = 32768                      # ≈64KB 有效 SRAM 按 fp16 计（A100 每 SM 164KB 同量级）
std, _ = hbm_access_std(N, d)
fa, _ = hbm_access_fa(N, d, M)
print('N=8192: 标准 HBM 访问 ≈ %.1f M 元素' % (std/1e6))
print('         FlashAttention ≈ %.1f M 元素' % (fa/1e6))
print('         理论减少 %.0f×（吞吐还受 kernel 效率/块尺寸影响）' % (std/fa))


N=8192: 标准 HBM 访问 ≈ 270.5 M 元素
         FlashAttention ≈ 51.4 M 元素
         理论减少 5×（吞吐还受 kernel 效率/块尺寸影响）


## 3️⃣ 在线 Softmax：FlashAttention 的核心数学

softmax 需要全局 max 做数值稳定，但分块时 max 未知 → **running max + rescale**：

$$ m^{(t)} = \max(m^{(t-1)},\ m_t), \qquad l^{(t)} = l^{(t-1)}e^{m^{(t-1)}-m^{(t)}} + \sum e^{x_t - m^{(t)}} $$

- 新块进来时旧累加项乘 $e^{m^{(t-1)}-m^{(t)}}$ 修正缩放 → **结果与全局 softmax 严格相等**；
- 同时解决数值溢出：$x$ 含 1000 时朴素 `exp(x)` 溢出，在线版安全；
- ⚠️ rescale 必须**同时**作用于已累加的分子/输出与 $l$：只更新 $l$ 而漏掉历史块的重缩放，就是最常见的在线 softmax 跨块 bug（下节代码已修复）。

In [2]:
def online_softmax(x, B):
    m, l = -np.inf, 0.0
    out = np.zeros(len(x))
    for i in range(0, len(x), B):
        xb = x[i:i+B]
        mb = xb.max()
        m_new = max(m, mb)
        alpha = np.exp(m - m_new)        # 跨块 rescale 系数 = exp(旧max - 新max)
        out = out*alpha                  # 之前所有块的 exp 分子同步重缩放（修复：漏掉则跨块不相等）
        out[i:i+B] += np.exp(xb - m_new) # 当前块的未归一化分子累加
        l = l*alpha + np.exp(xb - m_new).sum()  # l 与 out 用同一个 alpha 修正
        m = m_new
    return out/l, m, l

np.random.seed(0)
x = np.random.randn(1000)*10
ref = np.exp(x - x.max())/np.exp(x - x.max()).sum()
for B in (64, 256):
    o, m, l = online_softmax(x, B)
    assert np.allclose(o, ref, atol=1e-12), f'B={B} 分块必须与全局一致'
print('✓ 分块在线 softmax 与全局 softmax 完全一致')
big = np.array([1000.0, 1000.0, -999.0, 1000.0])
o, _, _ = online_softmax(big, 2)
print('✓ 朴素 exp(1000) 会溢出；在线版输出 =', np.round(o, 6))

✓ 分块在线 softmax 与全局 softmax 完全一致
✓ 朴素 exp(1000) 会溢出；在线版输出 = [0.333333 0.333333 0.       0.333333]


## 4️⃣ FlashAttention 前向：分块 + 在线更新

伪代码（按块循环 K/V 列，Q 块常驻 SRAM）：

```
for j in 块(K,V):
    S_j = Q_i K_j^T / √d          # 小块，不落 HBM
    m_new = max(m, rowmax(S_j))
    P_j = exp(S_j - m_new);  α = exp(m - m_new)
    l = l·α + rowsum(P_j)
    O = O·α + P_j V_j
    m = m_new
O = O / l                        # 统一归一化
```

![](images/10_flashattention_tiling_hbm_sram.svg)

**图 1**：左——标准注意力把 $S=QK^\top$、$P=\mathrm{softmax}(S)$ 两个 $N\times N$ 中间矩阵反复读写 HBM；右——FlashAttention 让 $Q$ 块常驻 SRAM，$K/V$ 按 tile 流式搬入，在线 $m,l$ 保证 $S,P$ 只存在于片上。

关键：**S、P 每次只存在 SRAM**，永远不写回 HBM → $N^2$ 项占用消失。

In [3]:
def flash_attn_fwd(q, k, v, Bc=2, scale=None):
    N, d = q.shape
    scale = scale or 1/np.sqrt(d)
    O = np.zeros_like(q)
    m = np.full(N, -np.inf); l = np.zeros(N)
    for j in range(0, N, Bc):               # K/V 分块
        kj, vj = k[j:j+Bc], v[j:j+Bc]
        s = (q @ kj.T)*scale                # 小块 S
        m_new = np.maximum(m, s.max(axis=1))
        p = np.exp(s - m_new[:, None])      # 小块 P
        alpha = np.exp(m - m_new)
        l = l*alpha + p.sum(axis=1)
        O = O*alpha[:, None] + p @ vj
        m = m_new
    return O/l[:, None], m, l

def standard_attn(q, k, v):
    s = (q @ k.T)/np.sqrt(q.shape[-1])
    p = np.exp(s - s.max(axis=1, keepdims=True))
    p = p/p.sum(axis=1, keepdims=True)
    return p @ v

np.random.seed(1)
N, d = 8, 8
q = np.random.randn(N, d); k = np.random.randn(N, d)
v = np.random.randn(N, d)*0.5
for Bc in (1, 2, 4):
    o, _, _ = flash_attn_fwd(q, k, v, Bc=Bc)
    err = np.abs(o - standard_attn(q, k, v)).max()
    print('Bc=%d 最大误差 = %.2e' % (Bc, err))
    assert err < 1e-9
print('✓ 分块 FlashAttention 前向与全量 softmax 数值等价')

Bc=1 最大误差 = 1.11e-16
Bc=2 最大误差 = 1.11e-16
Bc=4 最大误差 = 6.94e-17
✓ 分块 FlashAttention 前向与全量 softmax 数值等价


## 5️⃣ CUDA 线程模型：GPU 怎么跑这个循环

- **Grid → Block → Thread**：kernel 启动网格，每 block 一组线程（如 128/256 个），共享一块 **shared memory（SRAM）**；
- FlashAttention 布局：**每个 block 负责一行（或几行）Q**，把对应的 K/V 块搬进共享内存，块内线程并行算 S、P、O；
- **关键实践**：
  - 矩阵乘法用**分块 GEMM**（tiling）——数据留在共享内存反复复用；
  - 避免 **bank conflict**（共享内存同一周期多线程访问同 bank）；
  - 用向量化加载（float4）与**双缓冲**（计算当前块时预取下一块）隐藏延迟。

![](images/10_cuda_grid_block_thread.svg)

**图 2**：CUDA 执行层级 Grid → Block → Warp（32 线程）→ Thread：每个 block 负责一个 Q/K/V tile 并共享一块 shared memory，warp 是 SIMT 同步与 bank 访问的最小粒度。

In [4]:
# 模拟 CUDA tiling GEMM：输出 64×64，按 16×16 块分给 16 个 block
np.random.seed(2)
N, B = 64, 16
A = np.random.randn(N, N); C = np.random.randn(N, N)
D = np.zeros((N, N))
nb = N//B
for bi in range(nb):          # block 行
    for bj in range(nb):      # block 列
        acc = np.zeros((B, B))
        for bk in range(nb):  # 累加 K 维分块（共享内存命中）
            acc += A[bi*B:(bi+1)*B, bk*B:(bk+1)*B] @ C[bk*B:(bk+1)*B, bj*B:(bj+1)*B]
        D[bi*B:(bi+1)*B, bj*B:(bj+1)*B] = acc
print('tiling 结果与 numpy 全量一致: ', np.allclose(D, A @ C))
print('每个 block：累加 %d 次 %d×%d×%d 分块乘法（数据驻留共享内存）' % (nb, B, B, B))

tiling 结果与 numpy 全量一致:  True
每个 block：累加 4 次 16×16×16 分块乘法（数据驻留共享内存）


## 6️⃣ 反向传播：重计算（不存 S/P）

标准做法存 S、P 供反向用 → 8 GB 模型可能要吃几十 GB 激活；

FlashAttention 反向**不存中间量**：只需保存 $O$ 与 $l$（每行标量），反向时**重算** $S, P$：

- 前向存储：$O(Nd) + O(N)$（O 矩阵 + 行归一化常数）→ $O(N^2)$ 降到 $O(N)$ 量级；
- 反向计算量翻倍（~2× FLOP），但内存大幅下降，整体吞吐反而更高（少换入换出）；
- 这是「**用算力换内存带宽**」的典型工程取舍。

## 7️⃣ 变体与关联技术

- **FlashAttention-2**：消除非矩阵乘法开销（并行化行方向、更好的分块比例），HBM 访问减半，接近理论峰值；
- **Flash-Decoding**：长序列推理时，同一 query 对 KV 按块算局部输出再 reduce——把长 KV 分给多个 block 并行；
- **FlashAttention-3（2024）**：FP8、异步分块（warp specialization）再提速；
- 关联：**MQA/GQA**（减少 KV head → 少读重复数据）、**PagedAttention/vLLM**（KV 缓存分页）都与「注意力 IO 瓶颈」直接相关。

## 复杂度对比（背下来）

| 项 | 标准 | FlashAttention |
|---|---|---|
| 计算 FLOP | $O(N^2d)$ | $O(N^2d)$（相同） |
| HBM 访问 | $O(N^2 + Nd)$ | $O(N^2d^2/M + Nd)$（$M$=SRAM 元素数） |
| 额外内存 | $O(N^2)$（S、P） | $O(N)$（O、l） |
| 数值 | softmax 全局稳定 | 在线稳定，等价 |

面试金句：**FA 不减少 FLOP，减少的是 HBM 来回搬的字节数——带宽受限算子提速靠 IO 感知算法。**

In [5]:
# 不同序列长度下：标准 vs FA 的 HBM 访问量（元素数，对数尺度）
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
Ns = np.array([512, 1024, 2048, 4096, 8192, 16384])
d, M = 64, 32768
std = hbm_access_std(Ns, d)[0]
fa = hbm_access_fa(Ns, d, M)[0]
plt.figure(figsize=(8, 3.6))
plt.loglog(Ns, std, 'o-', label='标准注意力')
plt.loglog(Ns, fa, 's-', label='FlashAttention')
plt.xlabel('序列长度 N'); plt.ylabel('HBM 访问（元素）'); plt.legend()
plt.title('FA 的 HBM 访问与标准同为 O(N²)，但常数小 d²/M（≈5×），绝对差距随 N 增大')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_51124\2330097175.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 🎯 面试考点

1. 标准注意力内存为什么 O(N²)：S、P 写回 HBM，4N² 读写；
2. Roofline：什么是带宽受限；为什么 FA 是 IO-aware；
3. 在线 softmax：running max + rescale，为何与全局等价且防溢出；
4. 前向分块伪代码：S/P 只在 SRAM；
5. 反向重计算：省 O(N²) 存 S/P，用算力换带宽；
6. CUDA：grid/block/thread、shared memory、GEMM tiling、bank conflict；
7. FA-2/FA-3/Flash-Decoding 各自改进点；
8. 与 MQA/GQA、PagedAttention、长上下文的联系。

## ✅ 自测清单 + 衔接

- [ ] 手写在线 softmax 并与全局对照；
- [ ] 手写分块 flash_attn_fwd 多块尺寸验证等价；
- [ ] 手算标准/FA 的 HBM 访问量随 N 变化；
- [ ] 用 numpy 模拟 tiling GEMM；
- [ ] 说清反向重计算的存储收益与代价；
- [ ] 复述 8 条考点各 30 秒。

**衔接**：FlashAttention 是「推理引擎」的加速底座，与 07-量化、05-KV Cache、06-vLLM 一并构成大模型推理性能四件套。

## 8️⃣ 数值稳定性再探

- 连乘概率下溢 → 全程 log 域；
- `exp(x - max)` 前必须**先减 max**：`exp(1000)` 溢出为 inf、`exp(-1000)` 下溢为 0；
- 在线 softmax 的 rescale 系数 $e^{m^{(t-1)}-m^{(t)}}$ 始终保持 $\le 1$，不会放大误差；
- 结论：FA 的数值结果与全局 softmax **逐元素等价**（除浮点舍入）。

In [6]:
# 块尺寸（SRAM 容量）对 HBM 访问的影响
def fa_hbm(N, d, Bc):
    return 2*N**2*d/Bc + 2*N*d       # 教学近似：K/V 被重读 N/Bc 次

for N in (1024, 4096, 8192):
    std = 4*N**2
    row = 'N=%5d  标准 %.1fM →' % (N, std/1e6)
    for Bc in (64, 128, 256):
        row += '  FA(Bc=%d)=%.1fM' % (Bc, fa_hbm(N, 64, Bc)/1e6)
    print(row)
print('✓ SRAM 越大（Bc 越大），K/V 重读次数越少 → 提速')

N= 1024  标准 4.2M →  FA(Bc=64)=2.2M  FA(Bc=128)=1.2M  FA(Bc=256)=0.7M
N= 4096  标准 67.1M →  FA(Bc=64)=34.1M  FA(Bc=128)=17.3M  FA(Bc=256)=8.9M
N= 8192  标准 268.4M →  FA(Bc=64)=135.3M  FA(Bc=128)=68.2M  FA(Bc=256)=34.6M
✓ SRAM 越大（Bc 越大），K/V 重读次数越少 → 提速


## 9️⃣ FlashAttention-2 的三个改进

1. **Q 并行**：外层循环换成 Q 分块——各行 Q 完全独立并行（原版按 K 分块，Q 行并行受限）；
2. **去非乘法开销**：不用每个 block 内做 softmax 归一化的中间步骤，只保留必要的 rescale；
3. **更优分块比例**：$B_q \times B_k$ 与线程块匹配，寄存器复用最大化。

效果：HBM 访问再减半，在 A100 上达到接近理论峰值（~70%+ 的 MFU）。

In [7]:
def flash_attn_fwd2(q, k, v, Bq=2, Bk=2):
    # FA2 风格：外层 Q 分块（行并行），内层 K/V 分块累加
    N, d = q.shape
    scale = 1/np.sqrt(d)
    O = np.zeros_like(q)
    for i in range(0, N, Bq):
        qi = q[i:i+Bq]
        Oi = np.zeros_like(qi)
        mi = np.full(Bq, -np.inf); li = np.zeros(Bq)
        for j in range(0, N, Bk):
            s = (qi @ k[j:j+Bk].T)*scale
            m_new = np.maximum(mi, s.max(axis=1))
            p = np.exp(s - m_new[:, None])
            alpha = np.exp(mi - m_new)
            li = li*alpha + p.sum(axis=1)
            Oi = Oi*alpha[:, None] + p @ v[j:j+Bk]
            mi = m_new
        O[i:i+Bq] = Oi/li[:, None]
    return O

np.random.seed(3)
q = np.random.randn(8, 8); k = np.random.randn(8, 8); v = np.random.randn(8, 8)*0.5
o2 = flash_attn_fwd2(q, k, v, Bq=2, Bk=4)
err = np.abs(o2 - standard_attn(q, k, v)).max()
print('FA2 风格分块 vs 全量 softmax 最大误差 = %.2e' % err)
assert err < 1e-9
print('✓ FA2：Q 行分块并行 + K/V 内层分块，结果仍逐元素等价')

FA2 风格分块 vs 全量 softmax 最大误差 = 1.11e-16
✓ FA2：Q 行分块并行 + K/V 内层分块，结果仍逐元素等价


## 🔟 反向传播（梯度怎么回流）

记 $dO = \partial L/\partial O$，沿三条链式法则逐级回传：$O = PV$、$P = \mathrm{softmax}(S)$、$S = QK^\top/\sqrt d$。

**① 对 $O = PV$ 求梯度（$V$ 与 $P$）**

$$ \frac{\partial L}{\partial V} = P^\top dO, \qquad \frac{\partial L}{\partial P} = dO\,V^\top $$

**② softmax 的 Jacobian**：$\partial P_{il}/\partial S_{ij} = P_{il}(\delta_{lj} - P_{ij})$（同一行内自洽，行和为 0），得

$$ \frac{\partial L}{\partial S_{ij}} = P_{ij}\left(\left(\frac{\partial L}{\partial P}\right)_{ij} - \sum_l P_{il}\left(\frac{\partial L}{\partial P}\right)_{il}\right) $$

**③ 对 $S = QK^\top/\sqrt d$ 求梯度（$Q$ 与 $K$）**

$$ \frac{\partial L}{\partial Q} = \frac{1}{\sqrt d}\,\frac{\partial L}{\partial S}K, \qquad \frac{\partial L}{\partial K} = \frac{1}{\sqrt d}\left(\frac{\partial L}{\partial S}\right)^\top Q $$

合并简写：令 $h \triangleq \partial L/\partial P = dO\,V^\top$，每行只需一次行和 $\mathrm{rowsum}(P \odot h)$，四个梯度一步到位：

$$ dS_{ij} = P_{ij}\big(h_{ij} - \textstyle\sum_l P_{il} h_{il}\big), \qquad dQ = dS\,K/\sqrt d, \qquad dK = dS^\top Q/\sqrt d, \qquad dV = P^\top dO $$

- 反向需要 $P$（= softmax(S)）：**要么存（O(N²)），要么重算（+1 次前向 FLOP）**；
- FA 反向只存 $O$ 与行归一化常数 $l$，逐块重算 $S,P$ → 内存 $O(N)$，FLOP ~2×；
- 用算力换带宽，整体吞吐反而更高（少 HBM 往返）。

In [8]:
def softmax_attn(q, k, v):
    s = q@k.T/np.sqrt(q.shape[-1])
    p = np.exp(s - s.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    return p@v, p

def d_attn_grads(q, k, v, dL_dO):
    _, p = softmax_attn(q, k, v)
    h = dL_dO @ v.T                    # ∂L/∂P = dO·Vᵀ
    dS = p*(h - (p*h).sum(1, keepdims=True))   # softmax Jacobian
    s = 1/np.sqrt(q.shape[-1])
    return dS @ k*s, dS.T @ q*s, p.T @ dL_dO   # dQ, dK, dV

np.random.seed(4)
N, d = 4, 4
q = np.random.randn(N, d); k = np.random.randn(N, d)
v = np.random.randn(N, d); tg = np.random.randn(N, d)
o, _ = softmax_attn(q, k, v)
Gq, Gk, Gv = d_attn_grads(q, k, v, o - tg)   # 解析梯度 (dQ, dK, dV)

def loss_with(qq, kk, vv):
    oo, _ = softmax_attn(qq, kk, vv)
    return 0.5*((oo - tg)**2).sum()

eps = 1e-5
def num_grad(qq, kk, vv, name):
    key = name[-1].lower()                      # 'dQ'→'q', 'dK'→'k', 'dV'→'v'
    g = np.zeros_like({'q': qq, 'k': kk, 'v': vv}[key])
    for i in range(N):
        for j in range(d):
            pos = {'q': qq.copy(), 'k': kk.copy(), 'v': vv.copy()}
            neg = {'q': qq.copy(), 'k': kk.copy(), 'v': vv.copy()}
            pos[key][i, j] += eps; neg[key][i, j] -= eps
            g[i, j] = (loss_with(pos['q'], pos['k'], pos['v'])
                       - loss_with(neg['q'], neg['k'], neg['v']))/(2*eps)
    return g

for name, ga in (('dQ', Gq), ('dK', Gk), ('dV', Gv)):
    err = np.abs(ga - num_grad(q, k, v, name)).max()
    print('%-3s 解析 vs 数值梯度最大差 = %.2e' % (name, err))
    assert err < 1e-6
print('✓ dQ/dK/dV 解析式均经有限差分验证')


dQ  解析 vs 数值梯度最大差 = 1.64e-10
dK  解析 vs 数值梯度最大差 = 1.95e-10
dV  解析 vs 数值梯度最大差 = 1.42e-10
✓ dQ/dK/dV 解析式均经有限差分验证


## 1️⃣1️⃣ CUDA 内存模型与 bank conflict

```
Global(显存, 慢) → L2 → Shared(块内几十~几百KB, 快) → Register(最快)
```

- 共享内存在硬件上分成 **32 个 bank**（每 bank 4 字节），同一周期不同 bank 可并行访问；**所有线程访问同一地址是广播，不算冲突**；
- **bank conflict**：同一 warp 内 ≥2 线程访问**同一 bank 的不同地址** → 硬件串行化，慢 N 路。如 stride=8 的浮点数组，地址 $8i \bmod 32$ 只落进 4 个 bank（0/8/16/24），每 bank 落 8 次 → **8 路冲突，慢 8 倍**（stride=2 才是 2 路）；
- 解法：填充（padding）错开列偏移、换数据布局（SoA）、转置读取。

In [9]:
# bank conflict 模拟：同一 warp 32 线程的访问地址落在哪些 bank
BANKS = 32
def conflict_report(stride):
    addr = np.arange(32)*stride
    banks = addr % BANKS
    _, counts = np.unique(banks, return_counts=True)
    return counts.max(), int((counts > 1).sum())

for s in (1, 2, 8, 33):
    hot, n_conf = conflict_report(s)
    print('stride=%2d → 最热 bank 被访 %d 次, 冲突 bank %d 个' % (s, hot, n_conf))
print('✓ stride=1 无冲突；stride=2 每 bank 2 次（2-way）；stride=8 每 bank 8 次（8-way）；stride=33 互质 → 无冲突')
print('   （教学模拟；真实 CUDA 还需考虑地址位划分与类型宽度，double/float4 会跨 bank）')

stride= 1 → 最热 bank 被访 1 次, 冲突 bank 0 个
stride= 2 → 最热 bank 被访 2 次, 冲突 bank 16 个
stride= 8 → 最热 bank 被访 8 次, 冲突 bank 4 个
stride=33 → 最热 bank 被访 1 次, 冲突 bank 0 个
✓ stride=1 无冲突；stride=2 每 bank 2 次（2-way）；stride=8 每 bank 8 次（8-way）；stride=33 互质 → 无冲突
   （教学模拟；真实 CUDA 还需考虑地址位划分与类型宽度，double/float4 会跨 bank）


## 1️⃣2️⃣ Persistent Kernel 与 Warp 分工

- **Persistent Kernel**：固定少量 block 常驻 GPU，循环消费任务队列——省去反复启动 kernel 的开销（大模型逐层执行时尤其重要）；
- **Warp Specialization（FA3）**：不同 warp 分工——有的做 S/P 计算、有的预取下一块 K/V（**双缓冲**），隐藏 HBM 延迟；
- 收益：把「等数据」的时钟周期转化为「算数据」——FA3 在 H100 上再次提速 ~2×。

## 1️⃣3️⃣ 长上下文实战连线

- 长序列时 $N^2$ 项放大 → FA 的价值随 N 增长（128k 上下文必用）；
- **MQA/GQA**：减少 KV head 数 → 减少每个 query 重复读的 KV 字节（推理）；
- **PagedAttention（vLLM）**：KV 缓存按页管理，避免显存碎片 + 支持长上下文；
- 组合拳：GQA + FA + 分页 KV + 量化 = 现代推理引擎的四大件。

## 🎯 进阶追问集

- 「FA 和标准结果一样吗？」→ 逐元素等价（在线 softmax 的 rescale 是精确的）；
- 「为什么不直接加大 SRAM？」→ 容量与带宽的硬件约束，且 SRAM 贵；
- 「块尺寸怎么选？」→ 越大 HBM 访问越少，但要乘 4 倍矩阵 + 双缓冲，受 SRAM 限制；
- 「FA-2 为什么快一半？」→ 消除非乘法 FLOP + Q 行并行 + 分块比例匹配；
- 「训练和推理差别？」→ 训练还要反向重算（2×FLOP），推理只有前向但 KV 缓存/解码要单独优化。

In [10]:
# 大 logits 下朴素 softmax 直接炸：在线减 max 是关键
def softmax_naive(x): return np.exp(x)/np.exp(x).sum()
def softmax_online(x):
    m = x.max()
    e = np.exp(x - m)
    return e/e.sum()

x = np.array([800.0, 900.0, 850.0])        # 大 logits（常见于归一化前注意力）
na = softmax_naive(x)
ol = softmax_online(x)
print('朴素 softmax:', na, '（exp(900) 溢出 → 全 nan/inf）')
print('在线减 max :', np.round(ol, 6), '（正确）')
ref = np.exp(x - 900)/np.exp(x - 900).sum()
assert np.allclose(ol, ref)
print('✓ 面试点：FlashAttention 的 m 即在线 max，正确性靠它保证')

朴素 softmax: [nan nan nan] （exp(900) 溢出 → 全 nan/inf）
在线减 max : [0. 1. 0.] （正确）
✓ 面试点：FlashAttention 的 m 即在线 max，正确性靠它保证


C:\Users\24260\AppData\Local\Temp\ipykernel_51124\563939964.py:2: RuntimeWarning: overflow encountered in exp
  def softmax_naive(x): return np.exp(x)/np.exp(x).sum()
C:\Users\24260\AppData\Local\Temp\ipykernel_51124\563939964.py:2: RuntimeWarning: invalid value encountered in divide
  def softmax_naive(x): return np.exp(x)/np.exp(x).sum()


In [11]:
# 激活显存量化：标准注意力存 S/P（O(N²)） vs Flash（O(N)）
def std_act(N, d, bytes=4):
    return (N**2 + N**2 + N*d)*bytes      # S + P + O
def fa_act(N, d, bytes=4):
    return (N*d*2)*bytes                  # O + l/m

for N in (1024, 4096, 16384):
    print('N=%6d  d=64: 标准 %.1f MB | Flash %.2f MB | 省 %.0f×'
          % (N, std_act(N, 64)/1e6, fa_act(N, 64)/1e6, std_act(N, 64)/fa_act(N, 64)))
print('✓ N=16k 时标准要 2GB+ 激活，Flash 只要 8MB——长上下文训练必需')

N=  1024  d=64: 标准 8.7 MB | Flash 0.52 MB | 省 16×
N=  4096  d=64: 标准 135.3 MB | Flash 2.10 MB | 省 64×
N= 16384  d=64: 标准 2151.7 MB | Flash 8.39 MB | 省 256×
✓ N=16k 时标准要 2GB+ 激活，Flash 只要 8MB——长上下文训练必需


## 1️⃣4️⃣ GPU/CUDA 术语速查（面试默写）

- **SM**：流多处理器（计算单元）；一个 kernel 占多个 SM；
- **Warp**：32 线程同步执行单元（分支分叉则串行化）；
- **Occupancy**：SM 上活跃线程块比例（高=隐藏延迟好）；
- **HBM**：显存（带宽 GB/s 级）；**SRAM**：片上（带宽 TB/s 级）——FA 换的就是这个；
- **FLOPS**：峰值算力；**MFU**：实际/理论利用率（FA2 达到 ~70%+）；
- **向量化/双缓冲/流水线**：掩盖访存延迟的三大件。

## 🎤 AI-Infra 面试模拟问答（6 连）

Q「为什么 CUDA 要分 block/thread？」→ 层级并行 + 共享内存作用域（SM 内通信）；
Q「shared memory 多大？」→ 每 SM 48~228KB：A100 每 SM 164KB（块默认 48KB、动态上限 163KB），H100 每 SM 228KB——分块要精打细算；
Q「kernel 启动开销？」→ 微秒级，小计算不值得；persistent kernel 批量复用；
Q「训练慢先看什么？」→ 显存瓶颈→减小激活/梯度检查点；计算瓶颈→算子融合/重算；
Q「BF16 vs FP16？」→ BF16 动态范围大精度低（训练稳），FP16 精度高易溢出（需 loss scaling）；
Q「量化 4bit 掉点怎么补？」→ 权重/激活分离量化 + 校准 + 混合精度 + 蒸馏补偿。